#Log
* trim each column
* extract first 4 letter in prd_key as cat key as refference to yje catalog erp_px_cat_g1v2
* The next 6 letter in prd_key is the prd_key as refference to sales detail table
* replace null in prd_cst with 0
* normalization replace m,r,s,t in prd_line to Mountain, Road, Other Sales, Touring

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col
from pyspark.sql.window import Window

In [0]:
product=spark.table("workspace.bronze.crm_prd_info") 

product.display() 

In [0]:
null_counts_product=product.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in product.columns])

null_counts_product.show()

In [0]:
#trim every column
for field in product.schema.fields:
    if isinstance(field.dataType, StringType):
        product = product.withColumn(field.name, trim(col(field.name)))

#extract from prd_key to form cat_id(first 5) and prd_key(from 7 to end)
product = (product.withColumn("cat_id",F.substring(col("prd_key"),1,5))
           .withColumn("prd_key",F.substring(col("prd_key"),7,F.length(col("prd_key")))))

# replace null values in prd_cst as 0
product=product.fillna(0,['prd_cost'])

# product 
product =(product.withColumn("prd_line",
                             F.when(F.upper(F.col("prd_line"))=="M","Mountain")
                              .when(F.upper(F.col("prd_line"))=="R","Road")
                              .when(F.upper(F.col("prd_line"))=="S","Other Sales")
                              .when(F.upper(F.col("prd_line"))=="T","Touring")
                              .otherwise("Unknown")
                             )
)

# change prd_start_date to date time
product = product.withColumn("prd_start_dt",col("prd_start_dt").cast("date"))

# drop prd_end_dt

product = product.drop("prd_end_dt")

w=Window.partitionBy("prd_key").orderBy("prd_start_dt")

product = product.withColumn(
           "prd_end_dt",
           F.date_sub(F.lead("prd_start_dt").over(w),1).cast("date")
)

## move cat_id to third column plcae
col_order = product.columns
col_order.insert(1, col_order.pop(col_order.index("cat_id")))
product = product.select(col_order)

# rename columns
rename_map_product = {
    "prd_id":"prd_id",
    "cat_id":"category_id",
    "prd_key":"product_number",
    "prd_nm":"product_name",
    "prd_cost":"product_cost",
    "prd_line":"product_line",
    "prd_start_dt":"start_date",
    "prd_end_dt":"end_date"
}

for old_name, new_name in rename_map_product.items():
    product = product.withColumnRenamed(old_name, new_name)




In [0]:
product.display()

In [0]:
product.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")